In [0]:
# ========================================
# EXPLORATORY NOTEBOOK
# DATASET: gold_mart_customer_sales
# ========================================

# ========================================
# 0. CONFIGURATION
# ========================================

CATALOG = "ptfrozenfoods_dev"
SOURCE_SCHEMA = "gold"
TARGET_SCHEMA = "gold"

DOMAIN = "marts"
DATASET = "gold_mart_customer_sales"

STORAGE_ACCOUNT = "stptfrozenfoodsdevwe01"
GOLD_CONTAINER = "gold"

FACT_TABLE_NAME = "fact_sales"

FACT_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{FACT_TABLE_NAME}"

CANDIDATE_TARGET_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.mart_customer_sales"

GOLD_BASE_PATH = f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

FACT_SOURCE_PATH = f"{GOLD_BASE_PATH}facts/fact_sales/"
CANDIDATE_TARGET_PATH = f"{GOLD_BASE_PATH}marts/mart_customer_sales/"

In [0]:
# ========================================
# 1. CONTEXT SETUP
# ========================================

print("=" * 80)
print("STARTING GOLD EXPLORATORY: gold_mart_customer_sales")
print("=" * 80)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SOURCE_SCHEMA}")

print("[INFO] Context setup completed successfully.")

STARTING GOLD EXPLORATORY: gold_mart_customer_sales
[INFO] Context setup completed successfully.


In [0]:
# ========================================
# 2. CONFIGURATION SUMMARY
# ========================================

print("=" * 80)
print("GOLD EXPLORATORY NOTEBOOK CONFIGURATION")
print("=" * 80)
print(f"Catalog:                         {CATALOG}")
print(f"Source schema:                   {SOURCE_SCHEMA}")
print(f"Target schema:                   {TARGET_SCHEMA}")
print(f"Domain:                          {DOMAIN}")
print(f"Dataset:                         {DATASET}")
print(f"Fact table:                      {FACT_TABLE}")
print(f"Candidate target table:          {CANDIDATE_TARGET_TABLE}")
print(f"Fact source path:                {FACT_SOURCE_PATH}")
print(f"Candidate target path:           {CANDIDATE_TARGET_PATH}")
print("=" * 80)

GOLD EXPLORATORY NOTEBOOK CONFIGURATION
Catalog:                         ptfrozenfoods_dev
Source schema:                   gold
Target schema:                   gold
Domain:                          marts
Dataset:                         gold_mart_customer_sales
Fact table:                      ptfrozenfoods_dev.gold.fact_sales
Candidate target table:          ptfrozenfoods_dev.gold.mart_customer_sales
Fact source path:                abfss://gold@stptfrozenfoodsdevwe01.dfs.core.windows.net/facts/fact_sales/
Candidate target path:           abfss://gold@stptfrozenfoodsdevwe01.dfs.core.windows.net/marts/mart_customer_sales/


In [0]:
# ========================================
# 3. PRE-CHECKS
# ========================================

print("[INFO] Checking source table availability...")
spark.sql(f"DESCRIBE TABLE {FACT_TABLE}")

print("[INFO] Checking source path access...")
dbutils.fs.ls(FACT_SOURCE_PATH)

print("[INFO] Checking target container access...")
dbutils.fs.ls(f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/")

print("[INFO] Pre-checks completed successfully.")

[INFO] Checking source table availability...
[INFO] Checking source path access...
[INFO] Checking target container access...
[INFO] Pre-checks completed successfully.


In [0]:
# ========================================
# 4. READ SOURCE DATA
# ========================================

print("[INFO] Loading Gold source table...")

df_fact_sales = spark.table(FACT_TABLE)

print("[INFO] Source table loaded successfully.")

print("=" * 80)
print("SOURCE DATA ROW COUNTS")
print("=" * 80)
print(f"Fact sales:          {df_fact_sales.count():,}")
print("=" * 80)

[INFO] Loading Gold source table...
[INFO] Source table loaded successfully.
SOURCE DATA ROW COUNTS
Fact sales:          244,357


In [0]:
# ========================================
# DATASET PREVIEW — INITIAL EXPLORATION
# ========================================

print("=" * 80)
print("DATASET PREVIEW — INITIAL EXPLORATION")
print("=" * 80)

print("\n[INFO] Preview: FACT SALES")
display(df_fact_sales.limit(5))

print("\n[INFO] Printing schema...")
df_fact_sales.printSchema()

print("\n" + "=" * 80)
print("[INFO] Dataset preview completed successfully.")
print("=" * 80)

DATASET PREVIEW — INITIAL EXPLORATION

[INFO] Preview: FACT SALES


item_pedido_id,pedido_id,produto_id,cliente_id,canal_id,fornecedor_id,data_pedido,calendar_year,calendar_quarter,calendar_month,calendar_day,calendar_month_name,calendar_day_of_week,calendar_day_of_week_name,calendar_is_weekend,calendar_is_month_start,calendar_is_month_end,vendedor_id,cidade_entrega,estado_pedido,prazo_entrega_dias,tipo_cliente,cliente_cidade,distrito,segmento,potencial_valor,cluster_comercial,status_cliente,produto_nome,categoria,marca,status_produto,nome_canal,descricao_canal,quantity_sold,preco_lista_unitario,desconto_unitario,preco_venda_unitario,custo_unitario,gross_sales_amount,net_sales_amount,total_cost_amount,gross_margin_amount,line_count,flag_promocao,lote_fornecedor,item_load_date,item_ingestion_timestamp,item_source_file,order_load_date,order_ingestion_timestamp,order_source_file,average_order_value
IT000000027,PED2021011100012,P012,C0136,CH04,F006,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,UNKNOWN,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Hambúrguer bovino 800g,Carne,Chef Express,Ativo,Marketplace,Plataformas externas de venda,2,5.49,0.0,5.49,3.14,10.98,10.98,6.28,4.7,1,0,L66962,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,29.42
IT000000026,PED2021011100012,P020,C0136,CH04,F002,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,UNKNOWN,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Batata smile 1500g,Batatas,FrioMax,Ativo,Marketplace,Plataformas externas de venda,4,4.61,0.0,4.61,2.34,18.44,18.44,9.36,9.080000000000002,1,0,L39662,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,29.42
IT000000028,PED2021011100013,P004,C0136,CH02,F001,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,VEND001,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Bacalhau desfiado 800g,Peixe,PT Frozen Foods,Ativo,Vendas Externas,Equipa comercial visitando clientes,2,8.72,0.0,8.72,4.9,17.44,17.44,9.8,7.640000000000001,1,0,L28888,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,53.79
IT000000030,PED2021011100013,P023,C0136,CH02,F005,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,VEND001,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Pão de forma 800g,Padaria,Doce Norte,Ativo,Vendas Externas,Equipa comercial visitando clientes,4,2.97,0.15,2.82,1.36,11.88,11.28,5.44,5.839999999999999,1,1,L45385,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_orders/load_date=2026-03-17/erp_orders.csv,53.79
IT000000029,PED2021011100013,P020,C0136,CH02,F002,2021-01-11,2021,1,1,11,Janeiro,1,Segunda-feira,0,0,0,VEND001,Espinho,Entregue,1,Restaurante,Espinho,Aveiro,Pequeno,Alto,Cluster C,Ativo,Batata smile 1500g,Batatas,FrioMax,Ativo,Vendas Externas,Equipa comercial visitando clientes,3,4.93,0.0,4.93,2.24,14.79,14.79,6.720000000000001,8.069999999999999,1,0,L68711,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv,2026-03-17,2026-04-03T22:23:23.76Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.c


[INFO] Printing schema...
root
 |-- item_pedido_id: string (nullable = true)
 |-- pedido_id: string (nullable = true)
 |-- produto_id: string (nullable = true)
 |-- cliente_id: string (nullable = true)
 |-- canal_id: string (nullable = true)
 |-- fornecedor_id: string (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- calendar_year: integer (nullable = true)
 |-- calendar_quarter: integer (nullable = true)
 |-- calendar_month: integer (nullable = true)
 |-- calendar_day: integer (nullable = true)
 |-- calendar_month_name: string (nullable = true)
 |-- calendar_day_of_week: integer (nullable = true)
 |-- calendar_day_of_week_name: string (nullable = true)
 |-- calendar_is_weekend: integer (nullable = true)
 |-- calendar_is_month_start: integer (nullable = true)
 |-- calendar_is_month_end: integer (nullable = true)
 |-- vendedor_id: string (nullable = true)
 |-- cidade_entrega: string (nullable = true)
 |-- estado_pedido: string (nullable = true)
 |-- prazo_entrega_dias: in

In [0]:
# ========================================
# 5. SOURCE VALIDATION
# ========================================

from pyspark.sql import functions as F

required_fact_columns = [
    "pedido_id",
    "data_pedido",
    "cliente_id",
    "canal_id",
    "calendar_year",
    "calendar_quarter",
    "calendar_month",
    "calendar_day",
    "calendar_month_name",
    "calendar_day_of_week",
    "tipo_cliente",
    "cliente_cidade",
    "distrito",
    "segmento",
    "potencial_valor",
    "cluster_comercial",
    "status_cliente",
    "nome_canal",
    "descricao_canal",
    "quantity_sold",
    "gross_sales_amount",
    "net_sales_amount",
    "total_cost_amount",
    "gross_margin_amount",
    "line_count",
    "average_order_value"
]

missing_fact_columns = [c for c in required_fact_columns if c not in df_fact_sales.columns]

print(f"[INFO] Missing fact columns: {missing_fact_columns}")

if missing_fact_columns:
    raise Exception("Missing required columns in fact_sales.")

print("[INFO] Source validation completed successfully.")

[INFO] Missing fact columns: []
[INFO] Source validation completed successfully.


In [0]:
# ========================================
# 6. INITIAL DATA QUALITY CHECKS
# ========================================

display(
    df_fact_sales.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("data_pedido").alias("distinct_data_pedido"),
        F.countDistinct("cliente_id").alias("distinct_cliente_id"),
        F.countDistinct("canal_id").alias("distinct_canal_id"),
        F.sum(F.when(F.col("data_pedido").isNull(), 1).otherwise(0)).alias("null_data_pedido"),
        F.sum(F.when(F.col("cliente_id").isNull(), 1).otherwise(0)).alias("null_cliente_id"),
        F.sum(F.when(F.col("canal_id").isNull(), 1).otherwise(0)).alias("null_canal_id"),
        F.sum(F.when(F.col("net_sales_amount").isNull(), 1).otherwise(0)).alias("null_net_sales_amount"),
        F.sum(F.when(F.col("gross_margin_amount").isNull(), 1).otherwise(0)).alias("null_gross_margin_amount")
    )
)

total_rows,distinct_data_pedido,distinct_cliente_id,distinct_canal_id,null_data_pedido,null_cliente_id,null_canal_id,null_net_sales_amount,null_gross_margin_amount
244357,1875,145,4,0,0,0,0,0


In [0]:
# ========================================
# 7. BUSINESS PROFILE CHECKS
# ========================================

print("[INFO] Net sales by customer")
display(
    df_fact_sales.groupBy("cliente_id")
    .agg(
        F.round(F.sum("net_sales_amount"), 2).alias("total_net_sales"),
        F.round(F.sum("gross_margin_amount"), 2).alias("total_gross_margin"),
        F.sum("quantity_sold").alias("total_quantity_sold"),
        F.countDistinct("pedido_id").alias("total_orders")
    )
    .orderBy(F.desc("total_net_sales"))
)

print("[INFO] Net sales by customer segment")
display(
    df_fact_sales.groupBy("segmento")
    .agg(
        F.round(F.sum("net_sales_amount"), 2).alias("total_net_sales"),
        F.round(F.sum("gross_margin_amount"), 2).alias("total_gross_margin"),
        F.countDistinct("pedido_id").alias("total_orders")
    )
    .orderBy(F.desc("total_net_sales"))
)

print("[INFO] Net sales by customer type")
display(
    df_fact_sales.groupBy("tipo_cliente")
    .agg(
        F.round(F.sum("net_sales_amount"), 2).alias("total_net_sales"),
        F.round(F.sum("gross_margin_amount"), 2).alias("total_gross_margin"),
        F.countDistinct("pedido_id").alias("total_orders")
    )
    .orderBy(F.desc("total_net_sales"))
)

print("[INFO] Net sales by district")
display(
    df_fact_sales.groupBy("distrito")
    .agg(
        F.round(F.sum("net_sales_amount"), 2).alias("total_net_sales"),
        F.countDistinct("pedido_id").alias("total_orders"),
        F.sum("quantity_sold").alias("total_quantity_sold")
    )
    .orderBy(F.desc("total_net_sales"))
)

[INFO] Net sales by customer


cliente_id,total_net_sales,total_gross_margin,total_quantity_sold,total_orders
C0164,379267.11,143268.81,58090,4640
C0072,268120.63,105075.71,50331,3264
C0260,245366.82,95384.94,40651,2971
C0314,237682.05,91556.13,37234,3076
C0103,224874.99,90476.0,43001,2981
C0031,219373.69,88815.31,35372,2639
C0043,213785.07,83533.61,38796,2139
C0227,177809.19,71891.69,28636,2815
C0139,177150.45,70521.67,34414,2241
C0057,173566.39,68361.06,26536,3876


[INFO] Net sales by customer segment


segmento,total_net_sales,total_gross_margin,total_orders
Grande,3191628.39,1250190.72,40204
Médio,1716630.08,681642.7,33653
Pequeno,579080.79,227409.35,15669


[INFO] Net sales by customer type


tipo_cliente,total_net_sales,total_gross_margin,total_orders
Restaurante,2231920.02,861560.86,38966
Supermercado,1989603.52,790775.64,27672
Hotel,1085828.68,432161.35,15588
Take-away,165500.11,68437.25,6187
Particular,14486.93,6307.67,1113


[INFO] Net sales by district


distrito,total_net_sales,total_orders,total_quantity_sold
Porto,3287961.62,49421,553393
Braga,1246208.39,21156,207412
Aveiro,610447.56,12122,96920
Viana do Castelo,342721.69,6827,57068


In [0]:
# ========================================
# 8. BUILD CANDIDATE MART
# ========================================

df_mart_customer_sales_candidate = (
    df_fact_sales
    .groupBy(
        "data_pedido",
        "cliente_id",
        "canal_id"
    )
    .agg(
        F.round(F.sum("quantity_sold"), 2).alias("quantity_sold"),
        F.round(F.sum("gross_sales_amount"), 2).alias("gross_sales_amount"),
        F.round(F.sum("net_sales_amount"), 2).alias("net_sales_amount"),
        F.round(F.sum("total_cost_amount"), 2).alias("total_cost_amount"),
        F.round(F.sum("gross_margin_amount"), 2).alias("gross_margin_amount"),
        F.countDistinct("pedido_id").alias("order_count"),
        F.sum("line_count").alias("line_count"),
        F.round(F.avg("average_order_value"), 2).alias("average_order_value")
    )
)

print(f"[INFO] Candidate mart row count: {df_mart_customer_sales_candidate.count():,}")

display(df_mart_customer_sales_candidate.limit(10))

[INFO] Candidate mart row count: 71,234


data_pedido,cliente_id,canal_id,quantity_sold,gross_sales_amount,net_sales_amount,total_cost_amount,gross_margin_amount,order_count,line_count,average_order_value
2024-02-04,C0103,CH02,9,29.61,28.62,13.53,15.09,1,2,28.62
2024-06-03,C0072,CH02,53,369.8,342.54,203.72,138.82,3,12,115.53
2025-07-04,C0122,CH02,5,24.5,23.16,12.13,11.03,1,2,23.16
2025-07-22,C0314,CH01,19,110.39,103.75,63.4,40.35,1,5,103.75
2023-12-12,C0043,CH04,31,248.26,227.95,152.16,75.79,3,8,90.66
2022-01-22,C0293,CH02,13,86.81,83.7,50.92,32.78,2,5,48.75
2023-04-22,C0080,CH02,17,74.76,69.08,40.27,28.81,1,5,69.08
2024-05-08,C0164,CH01,19,145.4,140.18,88.58,51.6,1,4,140.18
2025-07-06,C0058,CH01,31,233.36,220.09,138.53,81.56,2,9,114.79
2024-12-19,C0256,CH02,38,144.61,130.05,78.92,51.13,1,6,130.05


In [0]:
# ========================================
# 9. CANDIDATE MART GRAIN TEST
# ========================================

print("[INFO] Testing candidate grain: one record per data_pedido, cliente_id, canal_id")

display(
    df_mart_customer_sales_candidate.select(
        F.count("*").alias("total_rows"),
        F.countDistinct(
            F.concat_ws("||", F.col("data_pedido"), F.col("cliente_id"), F.col("canal_id"))
        ).alias("distinct_grain_keys")
    )
)   

[INFO] Testing candidate grain: one record per data_pedido, cliente_id, canal_id


total_rows,distinct_grain_keys
71234,71234


In [0]:
# ========================================
# 10. CANDIDATE MART VALIDATION
# ========================================

display(
    df_mart_customer_sales_candidate.select(
        F.count("*").alias("total_rows"),
        F.countDistinct(
            F.concat_ws("||", F.col("data_pedido"), F.col("cliente_id"), F.col("canal_id"))
        ).alias("distinct_grain_keys"),
        F.sum(F.when(F.col("data_pedido").isNull(), 1).otherwise(0)).alias("null_data_pedido"),
        F.sum(F.when(F.col("cliente_id").isNull(), 1).otherwise(0)).alias("null_cliente_id"),
        F.sum(F.when(F.col("canal_id").isNull(), 1).otherwise(0)).alias("null_canal_id"),
        F.sum(F.when(F.col("net_sales_amount").isNull(), 1).otherwise(0)).alias("null_net_sales_amount"),
        F.sum(F.when(F.col("gross_margin_amount").isNull(), 1).otherwise(0)).alias("null_gross_margin_amount")
    )
)

print("[INFO] Candidate mart validation completed successfully.")

total_rows,distinct_grain_keys,null_data_pedido,null_cliente_id,null_canal_id,null_net_sales_amount,null_gross_margin_amount
71234,71234,0,0,0,0,0


[INFO] Candidate mart validation completed successfully.


In [0]:
# ========================================
# 11. METRIC RECONCILIATION CHECK
# ========================================

fact_totals = df_fact_sales.select(
    F.round(F.sum("quantity_sold"), 2).alias("fact_quantity_sold"),
    F.round(F.sum("gross_sales_amount"), 2).alias("fact_gross_sales_amount"),
    F.round(F.sum("net_sales_amount"), 2).alias("fact_net_sales_amount"),
    F.round(F.sum("total_cost_amount"), 2).alias("fact_total_cost_amount"),
    F.round(F.sum("gross_margin_amount"), 2).alias("fact_gross_margin_amount"),
    F.countDistinct("pedido_id").alias("fact_order_count"),
    F.sum("line_count").alias("fact_line_count")
)

candidate_totals = df_mart_customer_sales_candidate.select(
    F.round(F.sum("quantity_sold"), 2).alias("mart_quantity_sold"),
    F.round(F.sum("gross_sales_amount"), 2).alias("mart_gross_sales_amount"),
    F.round(F.sum("net_sales_amount"), 2).alias("mart_net_sales_amount"),
    F.round(F.sum("total_cost_amount"), 2).alias("mart_total_cost_amount"),
    F.round(F.sum("gross_margin_amount"), 2).alias("mart_gross_margin_amount"),
    F.sum("order_count").alias("mart_order_count"),
    F.sum("line_count").alias("mart_line_count")
)

display(fact_totals.crossJoin(candidate_totals))

fact_quantity_sold,fact_gross_sales_amount,fact_net_sales_amount,fact_total_cost_amount,fact_gross_margin_amount,fact_order_count,fact_line_count,mart_quantity_sold,mart_gross_sales_amount,mart_net_sales_amount,mart_total_cost_amount,mart_gross_margin_amount,mart_order_count,mart_line_count
914793,5780627.7,5487339.26,3328096.49,2159242.77,89526,244357,914793,5780627.7,5487339.26,3328096.49,2159242.77,89526,244357


In [0]:
# ========================================
# 12. MISSING VALUES ANALYSIS — MART CUSTOMER SALES
# ========================================

def analyze_missing_values(df, dataset_name):
    print("=" * 80)
    print(f"MISSING VALUES ANALYSIS — {dataset_name.upper()}")
    print("=" * 80)

    total_rows = df.count()

    missing_values_df = df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])

    missing_values_transposed = (
        missing_values_df
        .select(
            F.array([
                F.struct(
                    F.lit(col_name).alias("column_name"),
                    F.col(col_name).alias("null_count")
                ) for col_name in missing_values_df.columns
            ]).alias("missing_data")
        )
        .select(F.explode("missing_data").alias("data"))
        .select(
            F.col("data.column_name"),
            F.col("data.null_count")
        )
        .withColumn(
            "null_percentage",
            F.round((F.col("null_count") / F.lit(total_rows)) * 100, 4)
        )
        .orderBy(F.col("null_percentage").desc())
    )

    display(missing_values_transposed)

    print(f"[INFO] Total rows analyzed: {total_rows:,}")
    print("[INFO] Missing values analysis completed successfully.")
    print("=" * 80)

analyze_missing_values(df_mart_customer_sales_candidate, "mart_customer_sales")

MISSING VALUES ANALYSIS — MART_CUSTOMER_SALES


column_name,null_count,null_percentage
gross_margin_amount,0,0.0
gross_sales_amount,0,0.0
data_pedido,0,0.0
canal_id,0,0.0
net_sales_amount,0,0.0
cliente_id,0,0.0
line_count,0,0.0
order_count,0,0.0
total_cost_amount,0,0.0
quantity_sold,0,0.0


[INFO] Total rows analyzed: 71,234
[INFO] Missing values analysis completed successfully.


### Gold Layer Exploratory Conclusion — mart_customer_sales

This exploratory notebook validated the dataset required to build the `mart_customer_sales` analytical mart in the Gold layer of the PT Frozen Foods data platform. The objective was to confirm data quality, validate aggregation logic, and ensure readiness for analytical consumption.

---

### Grain Validation

The mart grain was defined as:

- **One record per (`data_pedido`, `cliente_id`, `canal_id`)**

Validation results:

- Total rows: 71,234
- Distinct grain keys: 71,234
- Null values: 0
- Duplicate values: 0

This confirms that the dataset is unique, consistent, and correctly aggregated.

---

### Business Value Assessment

This mart enables detailed analysis of customer behavior and sales performance across time and sales channels.

#### High-Value Analytical Metrics

- **Net Sales (`net_sales_amount`)**
  - Core metric for revenue analysis per customer.

- **Gross Margin (`gross_margin_amount`)**
  - Enables profitability analysis at customer level.

- **Quantity Sold (`quantity_sold`)**
  - Supports demand and volume analysis.

- **Order Count (`order_count`)**
  - Represents the number of distinct orders per customer, channel, and day.

- **Average Order Value (`average_order_value`)**
  - Provides insights into customer purchasing behavior.

#### Analytical Dimensions

- Customer (`cliente_id`)
- Sales Channel (`canal_id`)
- Date (`data_pedido`)

---

### Data Quality Assessment

The dataset demonstrated a high level of quality and consistency.

| Metric | Result |
|--------|--------|
| Total Records | 71,234 |
| Duplicate Records | 0 |
| Null Values | 0 |
| Schema Consistency | Validated |
| Grain Consistency | Confirmed |

---

### Metric Reconciliation

All additive metrics were successfully reconciled with the `fact_sales` table:

| Metric | Status |
|--------|--------|
| quantity_sold | Reconciled |
| gross_sales_amount | Reconciled |
| net_sales_amount | Reconciled |
| total_cost_amount | Reconciled |
| gross_margin_amount | Reconciled |
| line_count | Reconciled |

#### Order Count Behavior

- Fact distinct order count: 89,526  
- Mart aggregated order count: 89,526  

This confirms that `order_count` is fully additive at this grain, as each order is uniquely associated with a single customer and sales channel per day.

---

### Dimensional Model Alignment

The exploratory analysis confirms alignment with the analytical model:

- `fact_sales`
- `mart_customer_sales`

This mart is designed as a customer-centric analytical layer, enabling direct consumption for BI and advanced analytics.

---

### Medallion Architecture Alignment

| Layer | Status |
|-------|--------|
| Bronze | Completed |
| Silver | Completed |
| Gold | Validated |

---

### Governance and Technology Alignment

| Component | Technology |
|-----------|------------|
| Storage | ADLS Gen2 |
| Format | Delta Lake |
| Processing | Azure Databricks |
| Governance | Unity Catalog |
| Orchestration | Azure Data Factory |

---

### Key Decisions

| Decision | Outcome |
|----------|---------|
| Data source | Directly from enriched `fact_sales` |
| Join strategy | Removed unnecessary joins |
| Aggregation grain | Defined as (`data_pedido`, `cliente_id`, `canal_id`) |
| Metric reconciliation | Fully validated |
| Order count behavior | Confirmed as additive |
| Gold layer readiness | Approved |
| Performance optimization | Achieved through simplified processing |

---

### Conclusion

The dataset has been validated and approved for Gold layer processing. The analysis ensures:

- data quality and integrity;
- correct aggregation logic;
- full reconciliation with source data;
- adherence to analytical modeling best practices;
- improved performance through elimination of unnecessary joins;
- readiness for Business Intelligence and Machine Learning applications.

The platform is now ready for the implementation of the `mart_customer_sales` Gold processing notebook.